In [1]:
# Notebook: 03_decision_rule_operating_characteristics
# Monte Carlo operating characteristics of the Metric I decision rule, which labels
# a regime healthy, asymmetric, or defective from the signs of the per-arm
# directional separations (one-sided tests, Holm correction across the 2G
# comparisons, a materiality threshold delta). Three known regimes are simulated and
# the probability of correct classification is tracked against per-cell sample size,
# together with per-cell false-flag and miss rates.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RNG = np.random.default_rng(20260313)
BASE = os.path.join("..")
FIG = os.path.join(BASE, "results", "figures")
TAB = os.path.join(BASE, "results", "tables")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.30",
                     "axes.linewidth": 0.8, "grid.color": "0.88",
                     "grid.linewidth": 0.6, "savefig.bbox": "tight",
                     "figure.dpi": 150})


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, name + "." + ext), dpi=600)
    plt.close(fig)


rates = pd.read_csv(os.path.join(BASE, "data", "disclosed_rates.csv"))
rates["p"] = rates["default_rate_pct"] / 100.0
P = {(r.band, r.direction): (np.nan if pd.isna(r.p) else float(r.p))
     for r in rates.itertuples()}
BANDS = ["BB+", "BB", "BB-", "B+", "B", "B-"]
DELTA = 0.005          # materiality threshold on the default-rate scale (0.5 pp)
ALPHA = 0.05


# --- build three DGPs by transforming the disclosed hold rates ---
def make_regime(kind):
    # returns dict (band, dir) -> true default probability
    out = {}
    for g in BANDS:
        pH = P[(g, "H")]
        pD = P[(g, "D")]
        pU = P[(g, "U")]
        if kind == "asymmetric":
            out[(g, "U")] = pU   # disclosed: upgrade arm weak/deficient in low bands
            out[(g, "D")] = pD   # disclosed: downgrade arm value-adding
        elif kind == "healthy":
            out[(g, "U")] = pH * 0.55 if not np.isnan(pH) else np.nan  # upgrades default clearly less
            out[(g, "D")] = pH * 1.9                                   # downgrades default clearly more
        elif kind == "defective":
            out[(g, "U")] = pH * 1.9                                   # upgrades default MORE (reversed)
            out[(g, "D")] = pH * 0.55                                  # downgrades default LESS (reversed)
        out[(g, "H")] = pH
    return out


def one_sided_p(k1, n1, k0, n0, side):
    p1, p0 = k1 / n1, k0 / n0
    pp = (k1 + k0) / (n1 + n0)
    se = np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n0))
    if se == 0:
        return 1.0
    z = (p1 - p0) / se
    return float(stats.norm.sf(side * z))


def holm_reject(pairs, alpha):
    # pairs: list of (label, p_value); returns set of labels rejected under Holm
    m = len(pairs)
    order = sorted(pairs, key=lambda x: x[1])
    rejected = set()
    for i, (lab, pv) in enumerate(order):
        if pv <= alpha / (m - i):
            rejected.add(lab)
        else:
            break
    return rejected


def classify(sample, n):
    # sample: dict (band,dir)->default count at cell size n; apply the decision rule.
    # For each arm in each band, test for value-adding (sep > 0) and value-destroying
    # (sep < 0) with a materiality threshold, then aggregate to a regime label.
    tests_add, tests_destroy = [], []
    sep = {}
    for g in BANDS:
        kH = sample[(g, "H")]
        for d, side_label in (("U", "up"), ("D", "down")):
            if (g, d) not in sample or np.isnan(P.get((g, d), np.nan)) and sample[(g, d)] is None:
                continue
            kd = sample[(g, d)]
            if kd is None:
                continue
            # value-adding direction differs by arm: up adds value when p_U < p_H,
            # down adds value when p_D > p_H.
            if d == "U":
                sep_hat = (kH - kd) / n            # p_H - p_U ; >0 is value-adding
                p_add = one_sided_p(kH, n, kd, n, +1)
                p_destroy = one_sided_p(kd, n, kH, n, +1)  # p_U > p_H
            else:
                sep_hat = (kd - kH) / n            # p_D - p_H ; >0 is value-adding
                p_add = one_sided_p(kd, n, kH, n, +1)
                p_destroy = one_sided_p(kH, n, kd, n, +1)  # p_D < p_H
            sep[(g, d)] = sep_hat
            material = abs(sep_hat) >= DELTA
            tests_add.append((f"add:{g}:{d}", p_add if material and sep_hat > 0 else 1.0))
            tests_destroy.append((f"destroy:{g}:{d}", p_destroy if material and sep_hat < 0 else 1.0))
    add_rej = holm_reject(tests_add, ALPHA)
    destroy_rej = holm_reject(tests_destroy, ALPHA)
    up_add = sum(1 for lab in add_rej if lab.endswith(":U"))
    down_add = sum(1 for lab in add_rej if lab.endswith(":D"))
    up_destroy = sum(1 for lab in destroy_rej if lab.endswith(":U"))
    down_destroy = sum(1 for lab in destroy_rej if lab.endswith(":D"))
    # aggregation rule
    if up_destroy >= 2 or down_destroy >= 2:
        return "defective", (up_destroy, down_destroy)
    if up_add >= 2 and down_add >= 2:
        return "healthy", (up_destroy, down_destroy)
    return "asymmetric", (up_destroy, down_destroy)


N_GRID = [500, 1000, 2000, 5000, 10000, 20000]
R = 2500
REGIMES = ["asymmetric", "healthy", "defective"]
truth_probs = {k: make_regime(k) for k in REGIMES}

acc_rows, ff_rows = [], []
for kind in REGIMES:
    probs = truth_probs[kind]
    for n in N_GRID:
        correct = 0
        false_flag = 0           # a truly value-adding cell flagged value-destroying
        false_denom = 0
        for _ in range(R):
            sample = {}
            for g in BANDS:
                for d in ("U", "H", "D"):
                    p = probs[(g, d)]
                    sample[(g, d)] = None if (p is None or np.isnan(p)) else RNG.binomial(n, p)
            label, (upd, dwd) = classify(sample, n)
            if label == kind:
                correct += 1
            # false-flag audit on the downgrade arm when it is truly value-adding
            if kind in ("asymmetric", "healthy"):
                false_denom += 1
                if dwd >= 1:
                    false_flag += 1
        acc_rows.append({"regime": kind, "n_per_cell": n,
                         "accuracy": correct / R})
        if false_denom:
            ff_rows.append({"regime": kind, "n_per_cell": n,
                            "false_flag_rate_down_arm": false_flag / false_denom})
acc_df = pd.DataFrame(acc_rows)
ff_df = pd.DataFrame(ff_rows)
acc_df.to_csv(os.path.join(TAB, "t03_classification_accuracy.csv"), index=False)
ff_df.to_csv(os.path.join(TAB, "t03_false_flag_rate.csv"), index=False)

# --- figure 5: regime classification accuracy vs sample size (grayscale) ---
styles = {"asymmetric": ("0.10", "-", "o"),
          "healthy": ("0.40", "--", "s"),
          "defective": ("0.65", "-.", "^")}
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for kind in REGIMES:
    sub = acc_df[acc_df["regime"] == kind].sort_values("n_per_cell")
    c, ls, mk = styles[kind]
    ax.plot(sub["n_per_cell"], sub["accuracy"], color=c, linestyle=ls,
            marker=mk, markersize=5, markerfacecolor="white",
            markeredgecolor=c, linewidth=1.4, label=kind)
ax.axhline(0.90, color="0.55", linewidth=0.9, linestyle=(0, (1, 1)))
ax.set_xscale("log")
ax.set_xlabel("Firms per cell (log scale)")
ax.set_ylabel("Correct regime classification")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, loc="lower right", fontsize=9)
save_fig(fig, "fig05_decision_rule_accuracy")

print("Monte Carlo replications per point:", R)
print("\nClassification accuracy by regime:")
print(acc_df.pivot(index="n_per_cell", columns="regime", values="accuracy").round(3).to_string())
print("\nFalse-flag rate on a truly value-adding downgrade arm:")
print(ff_df.pivot(index="n_per_cell", columns="regime",
                  values="false_flag_rate_down_arm").round(3).to_string())


Monte Carlo replications per point: 2500

Classification accuracy by regime:
regime      asymmetric  defective  healthy
n_per_cell                                
500              1.000      0.157    0.002
1000             0.998      0.626    0.059
2000             0.994      0.988    0.517
5000             0.983      1.000    0.982
10000            0.989      1.000    0.998
20000            1.000      1.000    1.000

False-flag rate on a truly value-adding downgrade arm:
regime      asymmetric  healthy
n_per_cell                     
500              0.001      0.0
1000             0.000      0.0
2000             0.000      0.0
5000             0.000      0.0
10000            0.000      0.0
20000            0.000      0.0
